# Fit Model on full Dev Set

- Use model config derrived for each of self/parent cohorts in exploration to fit a single model
- Extract coefs from each trained self/parent model

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.residual_wrapper import get_df_path
from src.coef_visual import plot_heatmap
from src.utils import col_config_sensitivity
from src.residual_func import ResidualRegressor
import pandas as pd

In [ ]:
shared_dict = {
    "num_layers": 3,
    "l1_ratio2": None,
    "l1_ratio3": None,
    "max_iter": 50000,
    "seed": SEED,
    "impactme_outcome": None,
    "linear_type": "ridge",
    "pca_k": None,
    "pca_scale": None,
}
parent_config = {
    "impact_type": "parent",
    "quest_cols": col_config_sensitivity["parent_quest"],
    "agg_type": "max",
    "alpha2": 10.0,
    "alpha3": 5.0,
    "desc_seg_type": "pt_noshort",
    **shared_dict,
}
self_config = {
    "impact_type": "self",
    "quest_cols": col_config_sensitivity["self_quest"],
    "agg_type": "block",
    "alpha2": 40.0,
    "alpha3": 0.5,
    "desc_seg_type": "turns",
    **shared_dict,
}

In [ ]:
def build_model(config_dict, llama_model, data_dir):
    data_path = get_df_path(
        df_dir=data_dir,
        llama_model=llama_model,
        agg_type=config_dict["agg_type"],
        desc_seg_type=config_dict["desc_seg_type"],
        impact_type=config_dict["impact_type"],
    )
    train_df = pd.read_parquet(data_path)
    print(len(train_df))
    model = ResidualRegressor(**config_dict, two_layer_base=None)
    target_col = f"impact_score_{config_dict['impact_type']}"
    X_train = train_df.drop(target_col, axis=1)
    y_train = train_df[target_col]
    model.fit(X_train, y_train, shuffle=None, perm_dir=None)
    # Process coef df
    coef_df = model.get_coef_df()
    coef_df["stage_feat"] = coef_df["stage"] + "_" + coef_df["feature"]
    coef_df = coef_df.drop(columns=["stage", "feature"])
    # remove this so can combine self+parent
    replace_str = "p_dep" if config_dict["impact_type"] == "parent" else "s_dep"
    coef_df["stage_feat"] = coef_df["stage_feat"].str.replace(
        replace_str, "", regex=False
    )
    coef_df = coef_df.sort_values("stage_feat")
    coef_df = coef_df.set_index("stage_feat")
    return model, coef_df

In [ ]:
data_dir = BASE_PATH / "data"
llama_model = "Llama-3.3-70B-Instruct"
model_parent, coef_parent = build_model(parent_config, llama_model, data_dir)
model_self, coef_self = build_model(self_config, llama_model, data_dir)

In [ ]:
coef_parent = coef_parent.rename(columns={"coef": "coef_parent"})
coef_self = coef_self.rename(columns={"coef": "coef_self"})
coef_combined = coef_parent.join(coef_self)
plot_heatmap(
    coef_combined,
    title="Learned Coef weights for final parent and self models",
    vmin=-2.5,
    vmax=2.5,
    xlabel="Cohort",
)